In [1]:
import os
import json
import requests
import numpy as np
import pandas as pd

from openai import OpenAI
from typing import List
from pydantic import BaseModel, Field

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [13]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)
model_openai = "gpt-5.1"

In [4]:
# Example usage of querying similar listings
from query import query_similar_listings_example 
example_query = "Apartamento bonito con vista y adecuado para familias"
json_result = query_similar_listings_example(example_query, n_neighbors=5)
print("Similar listings to the query:")
print(json_result)

[DEBUG] Result columns:
Index(['id', 'name', 'price', 'description', 'listing_url', 'property_type',
       'room_type', 'neighbourhood_cleansed'],
      dtype='object')
Similar listings to the query:
[{"id":11572034,"name":"“Yaya’s Studio” 3BR\/3BA + Rooftop,garage,AC","price":8344.0,"description":"Step into luxury at our recently renovated 260 m2 \/townhouse- apartment. It's not just a stay; it's an experience that offers complete serenity and absolute privacy. Welcome to Yaya's Studio, nestled in the vibrant heart of Polanco","listing_url":"https:\/\/www.airbnb.com\/rooms\/11572034","property_type":"Entire rental unit","room_type":"Campsite","neighbourhood_cleansed":"Entire cottage"},{"id":28109161,"name":"Vita Polanco, Luxury Apartment, 250 MB Wi-Fi speed","price":1633.0,"description":"An elegant, cozy apartment with redundant internet services, including one with a 100 MB speed.<br \/><br \/>Over 100 reviews support our hosting quality. We are always open to suggestions for improv

In [5]:
query_similar_listings_example("Apartamento bonito con vista y adecuado para familias", n_neighbors=5)

[DEBUG] Result columns:
Index(['id', 'name', 'price', 'description', 'listing_url', 'property_type',
       'room_type', 'neighbourhood_cleansed'],
      dtype='object')


'[{"id":11572034,"name":"“Yaya’s Studio” 3BR\\/3BA + Rooftop,garage,AC","price":8344.0,"description":"Step into luxury at our recently renovated 260 m2 \\/townhouse- apartment. It\'s not just a stay; it\'s an experience that offers complete serenity and absolute privacy. Welcome to Yaya\'s Studio, nestled in the vibrant heart of Polanco","listing_url":"https:\\/\\/www.airbnb.com\\/rooms\\/11572034","property_type":"Entire rental unit","room_type":"Campsite","neighbourhood_cleansed":"Entire cottage"},{"id":28109161,"name":"Vita Polanco, Luxury Apartment, 250 MB Wi-Fi speed","price":1633.0,"description":"An elegant, cozy apartment with redundant internet services, including one with a 100 MB speed.<br \\/><br \\/>Over 100 reviews support our hosting quality. We are always open to suggestions for improvement, and treat all our guest with empathy, respect, and honesty. <br \\/><br \\/>We are a couple of owners, not a company that rents properties.<br \\/><br \\/>We inform our customers tha

In [7]:
get_similar_listings_json = {
    "name": "query_similar_listings_example",
    "description": "Usa esta herramienta (tool) para obtener los listings de airbnb que coinciden con la descripción proporcionada por el usuario. \
                    Evita busquedas ajenas a la herramienta proporcionada como url o recomendadores de internet.\
                    Contrasta siempre con las amenidades que también provee la herramienta para dar insights sin ser recomendación de compra o venta.",
    "parameters": {
        "type": "object",
        "properties": {
            "query_text": {
                "type": "string",
                "description": "La descripción en lenguaje natural para buscar listings similares."
            },
            "n_neighbors": {
                "type": "integer",
                "description": "Número de listings similares a retornar.",
                "default": 5
            }
        },
        "required": ["query_text"],
        "additionalProperties": False
    }
}


In [8]:
tools = [
    {"type": "function", "function": get_similar_listings_json}
]

In [9]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
system_prompt = "Eres un experto en recomendaciones de SmartBnB. Responde siempre en español."

In [14]:
def chat(message, history):
    client = OpenAI()
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = client.chat.completions.create(model="gpt-5.1", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason

        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [15]:
chat("¿Puedes ayudarme a encontrar un apartamento bonito con vista y adecuado para familias?", [])

Tool called: query_similar_listings_example
[DEBUG] Result columns:
Index(['id', 'name', 'price', 'description', 'listing_url', 'property_type',
       'room_type', 'neighbourhood_cleansed'],
      dtype='object')


'Con la descripción que diste, la herramienta encontró estas opciones que se acercan a lo que buscas (apartamento bonito, vistas y apto para familias). No son recomendaciones de compra/ reserva, pero sí ejemplos útiles para que sepas qué buscar y qué filtrar en Airbnb:\n\n1) **Blueground | Roma Sur 1 recámara, AC & rooftop**  \n- **Tipo**: Entire rental unit (departamento completo)  \n- **Lo más relevante**:  \n  - Menciona **“stunning balcony views over the city”**, es decir, muy buenas vistas desde el balcón.  \n  - Está **totalmente amueblado y equipado**, pensado para estadías más largas.  \n  - Suele ser ideal para parejas o familia muy pequeña (1 recámara).  \n- **Insight**: Si viajan con niños pequeños, revisa en el anuncio real:\n  - Capacidad de huéspedes  \n  - Si permite niños  \n  - Si el balcón es seguro (barandal alto, sin huecos peligrosos)\n\n2) **4 Regina. Apartment in the Historic Center**  \n- **Tipo**: Entire condo (departamento completo)  \n- **Lo más relevante**: 

In [8]:
from query import extract_pattern_availability
example_availability = extract_pattern_availability(listing_id=35797)
print("Extracted availability pattern:")
print(example_availability)

Extracted availability pattern:
{'listing_id': 35797, 'cluster_name': 'Siempre Disponible', 'intuition': 'Disponibilidad casi completa durante todo el año.'}


In [5]:
import sqlite3

# list tables from a sqlite database
def list_tables(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    conn.close()
    return [table[0] for table in tables]

db_path = "/Users/gblasd/Documents/SmartBnB/db/airbnb.db"
tables = list_tables(db_path)
print("Tables in the database:", tables)

Tables in the database: ['listing_ids', 'listings', 'reviews', 'calendar']
